# Notebook 3: Validate Exported Adapter Directly

Bu notebook tek bir adapter'i dogrudan yukler, metadata bilgisini okur, tek goruntu tahmini yapar ve isterseniz bir klasor uzerinde hizli saglik kontrolu calistirir.
Tek goruntu tahmini artik ayni dosyadan uretilen deterministik derived-view sonuclarini, opsiyonel occlusion sensitivity haritasini veya DINOv3 ViT attention map ciktisini da gosterebilir.

Notebook proje klasoru ve yaygin Drive koklerinde adapter bundle arar. Elle yol vermek de desteklenir.

Manuel giris alanlari:
- `ADAPTER_DIR`: adapter klasoru, export klasoru, telemetry run klasoru veya `adapter_meta.json`
- `ADAPTER_ROOT`: `models/adapters/` gibi crop klasorlerinin ebeveyni
- `IMAGE_PATH`: tek bir goruntu
- `BATCH_IMAGE_DIR`: bir klasor dolusu goruntu

Ayni oturum icinde tekrar denemek icin:
- `FORCE_ADAPTER_RESCAN = True` yapip adapter kesif hucresini tekrar calistirin
- tek goruntu hucresi varsayilan olarak yeni upload ister
- mevcut upload'i korumak icin `REUSE_EXISTING_IMAGE_PATH = True` yapin
- hemen yeni upload zorlamak icin `FORCE_IMAGE_UPLOAD = True` yapin


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb3_cell01_bootstrap_access.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb3_cell02_access_notice.py', globals())


In [ ]:
import json
from collections import Counter
from pathlib import Path

from IPython.display import HTML, display

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import ipywidgets as widgets
except Exception:
    widgets = None

from scripts.colab_adapter_smoke_test import (
    build_prediction_visualization_images,
    discover_adapter_candidates,
    load_adapter_summary,
    predict_image_folder,
    predict_single_image,
)

# Dogrudan smoke-test ayarlari.
CROP_NAME = None
# Ornek: 'tomato'. None birakirsaniz bulunan tum adapterler listelenir.

ADAPTER_DIR = None
# Kabul edilen ornekler:
# - .../continual_sd_lora_adapter/
# - .../artifacts/adapter_export/
# - outputs/colab_notebook_training/
# - .../telemetry/<RUN_ID>/ veya .../telemetry/<RUN_ID>/artifacts/
# - .../adapter_meta.json

ADAPTER_ROOT = None
# Normalde models/adapters/ gibi crop klasorlerinin ebeveyni olur.

SEARCH_ROOTS = [
    ROOT,
    ROOT / 'outputs',
    ROOT / 'models',
    ROOT / 'models' / 'adapters',
]
SELECTED_ADAPTER_INDEX = 0
FORCE_ADAPTER_RESCAN = False
IMAGE_PATH = None
FORCE_IMAGE_UPLOAD = False
# Tek goruntu dosyasi
BATCH_IMAGE_DIR = None
# Opsiyonel klasor testi icin goruntu klasoru
DEVICE = 'cuda'
CONFIG_ENV = 'colab'
REUSE_EXISTING_IMAGE_PATH = False
ENABLE_ROBUST_SMOKE = True
ROBUST_VIEWS = ('full_resize', 'resize_pad', 'center_crop')
ENABLE_PREDICTION_VISUALIZATION = True
EXPLANATION_METHOD = 'attention_map'
# Secenekler: 'attention_map' veya 'occlusion_sensitivity'
EXPLANATION_GRID_SIZE = 7

print(f'Cihaz={DEVICE} config_env={CONFIG_ENV}')
print('ADAPTER_DIR verirseniz kesif atlanir. Aksi halde notebook SEARCH_ROOTS altinda adapter arar.')
print('Kesif kokleri:')
for root in SEARCH_ROOTS:
    print(f'- {root}')
print('ADAPTER_ROOT crop klasorlerinin ebeveyni olmali, IMAGE_PATH tek dosya olmali, BATCH_IMAGE_DIR ise tek klasor olmali.')
print('Adapter listesini sifirdan kurmak icin FORCE_ADAPTER_RESCAN=True yapin.')
print('Tek goruntu hucresi varsayilan olarak yeni upload ister. Mevcut dosyayi korumak icin REUSE_EXISTING_IMAGE_PATH=True yapin.')
print(f'Robust smoke aktif={ENABLE_ROBUST_SMOKE} gorunumler={ROBUST_VIEWS}')
print(f'Gorsel aciklama aktif={ENABLE_PREDICTION_VISUALIZATION} method={EXPLANATION_METHOD} grid={EXPLANATION_GRID_SIZE}x{EXPLANATION_GRID_SIZE}')


In [ ]:
adapter_candidates = []
adapter_selector = None

if FORCE_ADAPTER_RESCAN:
    ADAPTER_DIR = None
    print('FORCE_ADAPTER_RESCAN=True, adapter listesi yeniden kuruluyor.')

if ADAPTER_DIR is not None:
    SELECTED_ADAPTER_DIR = str(Path(ADAPTER_DIR))
    print(f'Elle verilen ADAPTER_DIR kullaniliyor: {SELECTED_ADAPTER_DIR}')
else:
    adapter_candidates = discover_adapter_candidates(SEARCH_ROOTS, crop_name=CROP_NAME)
    if not adapter_candidates:
        raise FileNotFoundError(
            'SEARCH_ROOTS altinda adapter bulunamadi. Gerekirse ADAPTER_DIR degerini elle verin veya SEARCH_ROOTS listesini guncelleyin.'
        )

    print('Bulunan adapterler:')
    for index, candidate in enumerate(adapter_candidates):
        print(f'[{index}] {candidate["display_name"]}')

    if widgets is not None:
        adapter_selector = widgets.Dropdown(
            options=[(candidate['display_name'], index) for index, candidate in enumerate(adapter_candidates)],
            value=min(SELECTED_ADAPTER_INDEX, len(adapter_candidates) - 1),
            description='Adapter:',
            layout=widgets.Layout(width='95%'),
        )
        display(adapter_selector)
        print('Dropdown uzerinden adapter secin, sonra siradaki hucreyi calistirin.')
    else:
        print('ipywidgets yok. SELECTED_ADAPTER_INDEX degerini istediginiz adaptere ayarlayip sonraki hucreyi calistirin.')


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb3_cell05_adapter_summary.py', globals())


In [ ]:
request_new_upload = running_in_colab() and not bool(REUSE_EXISTING_IMAGE_PATH)
adapter_summary = globals().get('summary')
if (ADAPTER_DIR is None or CROP_NAME is None) and isinstance(adapter_summary, dict):
    ADAPTER_DIR = adapter_summary.get('resolved_adapter_dir', ADAPTER_DIR)
    CROP_NAME = adapter_summary.get('crop_name', CROP_NAME)

if ADAPTER_DIR is None and CROP_NAME is None:
    raise RuntimeError(
        'Once adapter secim/ozet hucresini calistirin ki Notebook 3 ADAPTER_DIR ve CROP_NAME degerlerini cozebilsin.'
    )

if FORCE_IMAGE_UPLOAD:
    IMAGE_PATH = None
    print('FORCE_IMAGE_UPLOAD=True, yeni goruntu yuklemesi isteniyor.')
elif request_new_upload:
    IMAGE_PATH = None
    print('REUSE_EXISTING_IMAGE_PATH=False, bu calisma icin yeni goruntu yuku istegi acildi.')

if IMAGE_PATH is None:
    if not running_in_colab():
        raise ValueError('Colab disinda calisiyorsaniz IMAGE_PATH degerini tek bir goruntu dosyasina ayarlayin.')
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError('Upload iptal edildi veya dosya secilmedi.')
    IMAGE_PATH = next(iter(uploaded.keys()))
    FORCE_IMAGE_UPLOAD = False

IMAGE_PATH = str(Path(IMAGE_PATH))
print(f'Kullanilan IMAGE_PATH: {IMAGE_PATH}')

single_result = predict_single_image(
    IMAGE_PATH,
    CROP_NAME,
    adapter_dir=ADAPTER_DIR,
    adapter_root=ADAPTER_ROOT,
    config_env=CONFIG_ENV,
    device=DEVICE,
    enable_robust_smoke=ENABLE_ROBUST_SMOKE,
    robust_views=ROBUST_VIEWS,
    explain_prediction=ENABLE_PREDICTION_VISUALIZATION,
    explanation_grid_size=EXPLANATION_GRID_SIZE,
    explanation_method=EXPLANATION_METHOD,
)

view_consistency = dict(single_result.get('view_consistency', {}))
uncertainty = dict(single_result.get('uncertainty_diagnostics', {}))

print('Notebook 3 notu: class confidence top-1 softmax skorudur; kalibre bir dogruluk olasiligi degildir.')
print('OOD karari ile class confidence ayni soruyu cevaplamaz; OOD dagilim uyumunu, confidence ise sinif tercihini ozetler.')

summary_rows = [
    {'metric': 'status', 'value': single_result.get('status')},
    {'metric': 'error', 'value': single_result.get('error')},
    {'metric': 'predicted_class', 'value': single_result.get('predicted_class')},
    {'metric': 'confidence', 'value': None if single_result.get('status') == 'error' else round(float(single_result.get('confidence', 0.0)), 4)},
    {'metric': 'is_ood', 'value': single_result.get('is_ood')},
    {'metric': 'score_method', 'value': single_result.get('score_method')},
    {'metric': 'primary_score', 'value': single_result.get('primary_score')},
    {'metric': 'decision_threshold', 'value': single_result.get('decision_threshold')},
    {'metric': 'primary_view', 'value': view_consistency.get('primary_view')},
    {'metric': 'stable_across_views', 'value': view_consistency.get('stable')},
]
if pd is not None:
    display(pd.DataFrame(summary_rows))
else:
    print(summary_rows)

view_rows = list(single_result.get('views', []))
if view_rows:
    print('Derived view results:')
    if pd is not None:
        display(pd.DataFrame(view_rows))
    else:
        print(view_rows)

visualization_images = build_prediction_visualization_images(IMAGE_PATH, single_result)
if visualization_images:
    print('Model gorunumu ve aciklama haritasi. Attention map attention routing bilgisidir; occlusion sensitivity parlak bolgeler kapatildiginda confidence dususunu gosterir.')
    display(visualization_images['model_view'])
    display(visualization_images['heatmap_overlay'])

print('Prediction status:', single_result.get('status'))
if single_result.get('error'):
    print('Prediction error:', single_result.get('error'))
print('View warnings:', view_consistency.get('warning_codes', []))
print('Uncertainty warnings:', uncertainty.get('warning_codes', []))
display(
    HTML(
        '<details><summary>Ham JSON</summary><pre>'
        + json.dumps(single_result, indent=2)
        + '</pre></details>'
    )
)


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb3_cell07_batch_prediction.py', globals())
